In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler,MinMaxScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest,f_regression
from sklearn.decomposition import PCA

In [3]:
df = pd.read_csv("01 covid_toy.csv")
display(df.head())

print("Dataset shape:", df.shape)

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


Dataset shape: (100, 6)


In [4]:
#Identifying numeric and categorical columns

numeric_columns = df.select_dtypes(include=np.number).columns.tolist()
categorical_columns = df.select_dtypes(include="object").columns.tolist()

print("Numeric Columns:")
print(numeric_columns)

print("\nCategorical Columns:")
print(categorical_columns)

Numeric Columns:
['age', 'fever']

Categorical Columns:
['gender', 'cough', 'city', 'has_covid']


In [5]:
#Checking for missing values

print("Missing values before cleaning:")
print(df.isnull().sum())

Missing values before cleaning:
age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64


In [6]:
# Replacing blank strings or question marks with missing values
df = df.replace(r"^\s*$", np.nan, regex=True)
df = df.replace("?", np.nan)

print("Missing values after checking blank values:")
print(df.isnull().sum())

Missing values after checking blank values:
age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64


In [7]:


df["fever"] = pd.to_numeric(df["fever"], errors="coerce")
df["fever"] = df["fever"].fillna(df["fever"].mean())

print("Missing values after filling fever:")
print(df.isnull().sum())

Missing values after filling fever:
age          0
gender       0
fever        0
cough        0
city         0
has_covid    0
dtype: int64


In [8]:

numeric_columns = df.select_dtypes(include=np.number).columns.tolist()

scaler = MinMaxScaler().set_output(transform="pandas")
scaled_data = scaler.fit_transform(df[numeric_columns])

display(scaled_data.head())

,age,fever
0,0.696203,0.833333
1,0.278481,0.333333
2,0.468354,0.500000
3,0.329114,0.000000
4,0.759494,0.500000


In [9]:
categorical_feature_columns = ["gender", "cough", "city"]

In [10]:
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
).set_output(transform="pandas")

encoded_data = encoder.fit_transform(df[categorical_feature_columns])

display(encoded_data.head())

,gender_Female,gender_Male,cough_Mild,cough_Strong,city_Bangalore,city_Delhi,city_Kolkata,city_Mumbai
0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0
1,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
2,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
3,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
4,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0


In [11]:
X = df.drop(columns=["has_covid"])
y = df["has_covid"]

In [14]:
X_numeric = X.select_dtypes(include=np.number)
y_numeric = y.map({'No': 0, 'Yes': 1})

selector = SelectKBest(
    score_func=f_regression,
    k=2
).set_output(transform="pandas")

X_selected = selector.fit_transform(X_numeric, y_numeric)

print(X_selected.head())

   age  fever
0   60  103.0
1   27  100.0
2   42  101.0
3   31   98.0
4   65  101.0
